# 07 – Events

ducto emits typed lifecycle events. Subscribe handlers to react to
credit operations.

Event types: `credits.added`, `credits.deducted`, `credits.refunded`,
`credits.low_balance`, `credits.cap_reached`, `credits.cap_warning`.

In [ ]:
from datetime import datetime, timedelta
from ducto.interface.postgres import PostgresStore
from ducto.manager import CreditManager
from ducto.engine import PricingEngine
from ducto.metrics import UsageMetrics, ToolCall
from ducto.interface.models import (
    PricingConfigData, PricingConfigV2, PlanDefinition,
    CreditMetadata,
)
from shared import start_postgres_store, cleanup

store, pgdata = start_postgres_store()
import uuid
from ducto.events import CreditEvent, CreditEventEmitter
print("✔ PostgresStore ready.")

### Create emitter and register handlers

In [ ]:
captured: list[CreditEvent] = []

def logger(ev: CreditEvent) -> None:
    captured.append(ev)
    print(f"  [{ev.type}] user={ev.user_id[:8]}…  data={ev.data}")

emitter = CreditEventEmitter()
emitter.on("credits.added", logger)
emitter.on("credits.deducted", logger)
emitter.on("credits.low_balance", logger)
print("Handlers registered.")

### Wire to CreditManager and trigger events

In [ ]:
manager = CreditManager(store, emitter=emitter)
user = str(uuid.uuid4())

print("--- add_credits ---")
manager.add_credits(user, 500)

print("\n--- deduct (end-to-end) ---")
engine = PricingEngine.from_dict({
    "version": 1,
    "models": {"_default": "input_tokens * 1"},
})
manager = CreditManager(store, engine=engine, emitter=emitter)
manager.add_credits(user, 2_000)
ded = manager.deduct(user, UsageMetrics(model="_default", input_tokens=100))
print(f"  Deduct result: amount={ded.amount}, balance={ded.balance_after}")

print(f"\nTotal events captured: {len(captured)}")

### Inspect captured events

In [ ]:
for ev in captured:
    print(f"  [{ev.timestamp.strftime('%H:%M:%S')}] {ev.type}")
    if ev.data:
        for k, v in ev.data.items():
            print(f"      {k}={v}")

### Subscribe by specific type

In [ ]:
refunds: list[CreditEvent] = []
emitter.on("credits.refunded", lambda e: refunds.append(e))

# trigger a refund — must use store directly
ded_tx = store.reserve_credits(user, 100, operation_type="test")
ded = store.deduct_credits(user, ded_tx.reservation_id, 100)
store.refund_credits(ded.transaction_id, amount=100, reason="demo")
print(f"Refund events: {len(refunds)}")

In [ ]:
cleanup(pgdata)